# HDB resale — XGBoost v1

**Model:** `XGBRegressor` on **`log1p(resale_price)`**; metrics and submission use **`expm1`** (dollar scale).

**Validation (two runs):**
1. **Time-based:** hold out the **last 12 calendar months** (`Tranc_YearMonth`).
2. **Random:** **70% / 30%** `train_test_split` (`shuffle=True`, `random_state=RNG`).

**Final submission:** refit on **full** training data using **`best_iteration` + 1** trees from the **time-based** model (matches early-stopped boosting rounds).

**Dependencies:** `pip install xgboost` or `conda install -c conda-forge py-xgboost`.

**Submissions:**
- `ROOT / submission / sub_xgb_v1_t.csv` (time split)
- `ROOT / submission / sub_xgb_v1_r.csv` (random split)



In [8]:
# Paths
from pathlib import Path

import numpy as np
import pandas as pd

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebook" else _cwd
TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUB_PATH = ROOT / "data" / "sample_sub_reg.csv"
SUBMISSION_PATH_T = ROOT / "submission" / "sub_xgb_v1_t.csv"
SUBMISSION_PATH_R = ROOT / "submission" / "sub_xgb_v1_r.csv"

RNG = 42

print(f"ROOT: {ROOT.resolve()}")
train = pd.read_csv(TRAIN_PATH, low_memory=False)
test = pd.read_csv(TEST_PATH, low_memory=False)
print(train.shape, test.shape)



ROOT: /Users/ian/Documents/NTU/DSAI/Module 3/HDB Kaggle
(150634, 77) (16735, 76)


In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

from xgboost import XGBRegressor

TARGET = "resale_price"



In [ ]:
ROOMS_FROM_FLAT = {
    "1 ROOM": 1,
    "2 ROOM": 2,
    "3 ROOM": 3,
    "4 ROOM": 4,
    "5 ROOM": 5,
    "EXECUTIVE": 6,
    "MULTI-GENERATION": 7,
}


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    pc = out["postal"].astype(str).str.replace(r"\.0$", "", regex=True)
    out["postal_sector"] = pd.to_numeric(pc.str.slice(0, 2), errors="coerce")

    ms = pd.to_numeric(out["mid_storey"], errors="coerce")
    mx = pd.to_numeric(out["max_floor_lvl"], errors="coerce")
    out["storey_ratio"] = np.where(mx > 0, ms / mx, np.nan)

    rcols = ["1room_rental", "2room_rental", "3room_rental", "other_room_rental"]
    total_rent = np.zeros(len(out))
    for c in rcols:
        total_rent += pd.to_numeric(out[c], errors="coerce").fillna(0).to_numpy(dtype=float)
    td = pd.to_numeric(out["total_dwelling_units"], errors="coerce").to_numpy(dtype=float)
    out["rental_ratio"] = np.where(td > 0, total_rent / td, np.nan)

    out["rooms_num"] = out["flat_type"].map(ROOMS_FROM_FLAT).astype(float)
    ty = pd.to_numeric(out["Tranc_Year"], errors="coerce")
    tm = pd.to_numeric(out["Tranc_Month"], errors="coerce")
    out["month_index"] = (ty - 2000) * 12 + tm
    return out


train = add_engineered_features(train)
test = add_engineered_features(test)

DROP_FEATURES = [
    "id",
    "Tranc_YearMonth",
    "Tranc_Year",
    "Tranc_Month",
    "floor_area_sqft",
    "postal",
    "address",
    "block",
    "street_name",
    "flat_type",
    "flat_model",
    "1room_sold",
    "2room_sold",
    "3room_sold",
    "4room_sold",
    "5room_sold",
    "exec_sold",
    "multigen_sold",
    "studio_apartment_sold",
    "1room_rental",
    "2room_rental",
    "3room_rental",
    "other_room_rental",
    "bus_stop_name",
    "sec_sch_name",
    
]

feature_cols = [c for c in train.columns if c not in DROP_FEATURES and c != TARGET]
assert not set(feature_cols) - set(test.columns)

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y = train[TARGET].astype(float)

period = pd.to_datetime(train["Tranc_YearMonth"], format="%Y-%m")

print(
    f"Features: {len(feature_cols)} (incl. month_index from Tranc_Year/Tranc_Month)"
)



Features: 60 (incl. month_index from Tranc_Year/Tranc_Month)


In [34]:
def imputation_stats(X_ref: pd.DataFrame, cat_cols: list, num_cols: list):
    num_med = {
        c: pd.to_numeric(X_ref[c], errors="coerce").median()
        for c in num_cols
    }
    cat_fill = {}
    for c in cat_cols:
        s = X_ref[c].astype(str).replace("nan", np.nan)
        m = s.mode(dropna=True)
        cat_fill[c] = m.iloc[0] if len(m) else "_MISSING_"
    return num_med, cat_fill


def prepare_xgb(
    X: pd.DataFrame,
    cat_cols: list,
    num_cols: list,
    num_med: dict,
    cat_fill: dict,
    cat_category_lists=None,
) -> pd.DataFrame:
    """Numeric medians; categoricals as pandas category for XGBoost native cat support."""
    out = X.copy()
    for c in num_cols:
        v = pd.to_numeric(out[c], errors="coerce")
        out[c] = v.fillna(num_med[c]).astype(float)
    for c in cat_cols:
        s = out[c].astype(str).replace("nan", np.nan).fillna(cat_fill[c])
        if cat_category_lists is not None:
            cats = cat_category_lists[c]
            allowed = set(cats)
            s = s.where(s.isin(allowed), cat_fill[c])
            out[c] = pd.Categorical(s, categories=cats)
        else:
            out[c] = s.astype("category")
    return out


num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]
print(f"Numeric: {len(num_cols)}, categorical (XGBoost): {len(cat_cols)}")



Numeric: 47, categorical (XGBoost): 13


In [27]:
# --- A) Time-based validation (last 12 months) ---
cutoff = period.max() - pd.DateOffset(months=12)
tr_time = period < cutoff
va_time = ~tr_time

X_tr_raw = X_train.loc[tr_time]
X_va_raw = X_train.loc[va_time]
y_tr_t = y.loc[tr_time]
y_va_t = y.loc[va_time]

num_med_t, cat_fill_t = imputation_stats(X_tr_raw, cat_cols, num_cols)
X_tr_t = prepare_xgb(X_tr_raw, cat_cols, num_cols, num_med_t, cat_fill_t, cat_category_lists=None)
cat_lists_t = {c: list(X_tr_t[c].cat.categories) for c in cat_cols}
X_va_t = prepare_xgb(X_va_raw, cat_cols, num_cols, num_med_t, cat_fill_t, cat_category_lists=cat_lists_t)

y_tr_log_t = np.log1p(y_tr_t.values)
y_va_log_t = np.log1p(y_va_t.values)

model_time = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=RNG,
    early_stopping_rounds=80,
    eval_metric="rmse",
    enable_categorical=True,
    tree_method="hist",
    n_jobs=-1,
)
model_time.fit(
    X_tr_t,
    y_tr_log_t,
    eval_set=[(X_va_t, y_va_log_t)],
    verbose=False,
)

pred_va_t = np.expm1(model_time.predict(X_va_t))
rmse_t = root_mean_squared_error(y_va_t, pred_va_t)
mae_t = mean_absolute_error(y_va_t, pred_va_t)
print(
    f"Time split: train {len(X_tr_t):,}, val {len(X_va_t):,} (val >= {cutoff.date()})"
)
print(f"Time-split validation RMSE (dollars): {rmse_t:,.2f}")
print(f"Time-split validation MAE (dollars): {mae_t:,.2f}")
bi = getattr(model_time, "best_iteration", None)
print(f"time model best_iteration (0-based): {bi}")



Time split: train 128,899, val 21,735 (val >= 2020-04-01)
Time-split validation RMSE (dollars): 36,487.14
Time-split validation MAE (dollars): 26,357.88
time model best_iteration (0-based): 466


In [35]:
# --- B) Random 80% / 20% validation ---
X_tr_raw_r, X_va_raw_r, y_tr_r, y_va_r = train_test_split(
    X_train,
    y,
    test_size=0.20,
    random_state=RNG,
    shuffle=True,
)

num_med_r, cat_fill_r = imputation_stats(X_tr_raw_r, cat_cols, num_cols)
X_tr_r = prepare_xgb(X_tr_raw_r, cat_cols, num_cols, num_med_r, cat_fill_r, cat_category_lists=None)
cat_lists_r = {c: list(X_tr_r[c].cat.categories) for c in cat_cols}
X_va_r = prepare_xgb(X_va_raw_r, cat_cols, num_cols, num_med_r, cat_fill_r, cat_category_lists=cat_lists_r)

y_tr_log_r = np.log1p(y_tr_r.values)
y_va_log_r = np.log1p(y_va_r.values)

model_rand = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=RNG,
    early_stopping_rounds=80,
    eval_metric="rmse",
    enable_categorical=True,
    tree_method="hist",
    n_jobs=-1,
)
model_rand.fit(
    X_tr_r,
    y_tr_log_r,
    eval_set=[(X_va_r, y_va_log_r)],
    verbose=False,
)

pred_va_r = np.expm1(model_rand.predict(X_va_r))
rmse_r = root_mean_squared_error(y_va_r, pred_va_r)
mae_r = mean_absolute_error(y_va_r, pred_va_r)
print(
    f"Random 80/20: train {len(X_tr_r):,} ({100 * len(X_tr_r) / len(X_train):.1f}%), "
    f"val {len(X_va_r):,} ({100 * len(X_va_r) / len(X_train):.1f}%)"
)
print(f"Random-split validation RMSE (dollars): {rmse_r:,.2f}")
print(f"Random-split validation MAE (dollars): {mae_r:,.2f}")
print(f"random model best_iteration: {getattr(model_rand, 'best_iteration', None)}")



Random 80/20: train 120,507 (80.0%), val 30,127 (20.0%)
Random-split validation RMSE (dollars): 23,592.38
Random-split validation MAE (dollars): 16,773.89
random model best_iteration: 612


In [14]:
# --- C) Full train refit for submissions (_t and _r) ---
bi_t = getattr(model_time, "best_iteration", None)
if bi_t is None:
    n_trees_t = 500
else:
    n_trees_t = int(bi_t) + 1
if n_trees_t < 50:
    n_trees_t = 500

bi_r = getattr(model_rand, "best_iteration", None)
if bi_r is None:
    n_trees_r = 500
else:
    n_trees_r = int(bi_r) + 1
if n_trees_r < 50:
    n_trees_r = 500

num_med_f, cat_fill_f = imputation_stats(X_train, cat_cols, num_cols)
X_full = prepare_xgb(X_train, cat_cols, num_cols, num_med_f, cat_fill_f, cat_category_lists=None)
cat_lists_f = {c: list(X_full[c].cat.categories) for c in cat_cols}
X_test_p = prepare_xgb(X_test, cat_cols, num_cols, num_med_f, cat_fill_f, cat_category_lists=cat_lists_f)
y_log_full = np.log1p(y.values)

model_final_t = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=n_trees_t,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=RNG,
    enable_categorical=True,
    tree_method="hist",
    n_jobs=-1,
)
model_final_r = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=n_trees_r,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=RNG,
    enable_categorical=True,
    tree_method="hist",
    n_jobs=-1,
)

model_final_t.fit(X_full, y_log_full, verbose=False)
model_final_r.fit(X_full, y_log_full, verbose=False)

test_pred_t = np.expm1(model_final_t.predict(X_test_p))
test_pred_r = np.expm1(model_final_r.predict(X_test_p))

sample = pd.read_csv(SAMPLE_SUB_PATH, nrows=5)
sub_t = pd.DataFrame({"Id": test["id"], "Predicted": test_pred_t})
sub_r = pd.DataFrame({"Id": test["id"], "Predicted": test_pred_r})
assert list(sub_t.columns) == list(sample.columns)
assert list(sub_r.columns) == list(sample.columns)

SUBMISSION_PATH_T.parent.mkdir(parents=True, exist_ok=True)
sub_t.to_csv(SUBMISSION_PATH_T, index=False)
sub_r.to_csv(SUBMISSION_PATH_R, index=False)
print(f"Final time model n_estimators (from time best_iteration + 1): {n_trees_t}")
print(f"Final random model n_estimators (from random best_iteration + 1): {n_trees_r}")
print(f"Wrote {SUBMISSION_PATH_T.resolve()} ({len(sub_t):,} rows)")
print(f"Wrote {SUBMISSION_PATH_R.resolve()} ({len(sub_r):,} rows)")
print(sub_t.head())


Final time model n_estimators (from time best_iteration + 1): 670
Final random model n_estimators (from random best_iteration + 1): 701
Wrote /Users/ian/Documents/NTU/DSAI/Module 3/HDB Kaggle/submission/sub_xgb_v1_t.csv (16,735 rows)
Wrote /Users/ian/Documents/NTU/DSAI/Module 3/HDB Kaggle/submission/sub_xgb_v1_r.csv (16,735 rows)
       Id     Predicted
0  114982  375079.56250
1   95653  450596.43750
2   40303  348890.40625
3  109506  299334.31250
4  100149  411698.93750


## Compare splits

- **Random 70/30** RMSE is often **lower** than the **time-based** RMSE because future rows can appear in the training fold.
- Submission files now include both variants:
  - `_t.csv` uses trees from the time-split early-stopped model.
  - `_r.csv` uses trees from the random-split early-stopped model.
